# Phase 3 — Generation Test
**FSQA** | fine-tune pipeline

> **Bring your own data.** No data ships with this repo. This notebook reads `./local-data/phase1_fsqa_eval/<run id>/questions.jsonl` (JSON Lines, one question per line, as Phase 1 writes it) and `./local-data/phase2_splits/split_<Phase 1 run id>.json` (the A/B split Phase 2b writes). It uses the last Phase 1 run folder in name order, and the baseline pairs are the set B questions from the six seeds in the config cell. Run Phases 1 and 2b first, or supply your own files in those formats. Each question record needs `id`, `batch_id` (matching a `batch_id` in `SEEDS`), `seed_section`, `domain`, `question`, `reference_answer` and `citation`; the split file needs `set_b` (a list of question ids).

Everything before this measured **answering**. Phase 4 needs **generating** —
inventing a realistic scenario, grounding it in a CFR section, and writing a
correct answer. Different skill, never tested.

**Design**
- `gpt-5.5|xhigh` generates pairs using Phase 1's exact prompt on six seeds
  drawn from set B batches.
- The Opus 5 pairs from those same six seeds are the baseline. Same sections,
  same archetypes, so section difficulty cannot confound the comparison.
- Pairs are **shuffled and stripped of their source** before grading. Judges
  see an anonymous id.
- Four judges: Gemini 3.1 Pro, Kimi K3, Opus 5, Sol 5.6 xhigh.

**On Opus grading:** Opus wrote the baseline pairs, so it grades its own
output here. That is why it is one of four and why the notebook reports each
judge's score for each source separately — self-preference shows up as Opus
rating Opus higher than the other three do.

**On Sol grading:** Sol and 5.5 are both OpenAI. Same test applies; if Sol
rates 5.5's pairs above the other judges' average, that is vendor affinity
and it is measured rather than assumed.

**Holdout note:** this exposes the baseline set B pairs to the judges. The
candidate models never see them, but Sol does. Their ids are recorded so the
final evaluation can exclude them.

In [ ]:
import concurrent.futures
import itertools
import json
import os
import random
import re
import statistics
import threading
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone

import requests

print("imports ok", flush=True)

## Config

In [ ]:
GENERATOR = {"config_id": "gpt-5.5|xhigh", "model": "openai/gpt-5.5", "effort": "xhigh"}

JUDGES = {
    "gemini": {"slug": "google/gemini-3.1-pro-preview",
               "provider": {"order": ["google-vertex", "google-ai-studio"], "allow_fallbacks": True},
               "reasoning": None},
    "kimi":   {"slug": "moonshotai/kimi-k3",
               "provider": {"order": ["digitalocean"], "allow_fallbacks": False},
               "reasoning": None},
    "opus":   {"slug": "anthropic/claude-opus-5",
               "provider": {"order": ["amazon-bedrock"], "allow_fallbacks": False},
               "reasoning": None},
    "sol":    {"slug": "openai/gpt-5.6-sol",
               "provider": {"order": ["openai"], "allow_fallbacks": False},
               "reasoning": {"effort": "xhigh"}},
}
USE_JUDGES = ["gemini", "kimi", "opus", "sol"]

EVAL_PROVIDER = {"order": ["openai"], "allow_fallbacks": False}

# Confirmed listed prices, USD per million tokens (input, output). Source for every entry:
# https://openrouter.ai/api/v1/models/<model slug>/endpoints, retrieved 2026-11-05. Each figure is the highest
# price across that model's endpoints and tiers, so reservations are worst case.
PRICE = {
    "openai/gpt-5.5": (25.00, 112.50),                  # OpenAI Fast (priority) tier, prompts over 272k tokens
    "google/gemini-3.1-pro-preview": (7.20, 32.40),     # priority tier, prompts over 200k tokens
    "moonshotai/kimi-k3": (4.50, 22.50),                # Parasail/fast and Fireworks/fast priority endpoints
    "anthropic/claude-opus-5": (10.00, 50.00),          # Anthropic fast/priority tier
    "openai/gpt-5.6-sol": (8.80, 33.00),                # regional endpoints, prompts over 272k tokens
}

# Six matched seeds: batch_id -> (section, archetype). Pick seeds for which your
# set B holds enough Phase 1 pairs to serve as a baseline.
SEEDS = [
    {"batch_id": "fsvp-016",   "domain": "fsvp",   "section": "1.500",   "archetype": "procedural sequence"},
    {"batch_id": "fsvp-020",   "domain": "fsvp",   "section": "1.504",   "archetype": "conflict between two provisions"},
    {"batch_id": "recall-012", "domain": "recall", "section": "810.11",  "archetype": "timeline or deadline"},
    {"batch_id": "recall-011", "domain": "recall", "section": "810.10",  "archetype": "applicability"},
    {"batch_id": "pchf-001",   "domain": "pchf",   "section": "117.3",   "archetype": "applicability"},
    {"batch_id": "pchf-002",   "domain": "pchf",   "section": "117.5",   "archetype": "timeline or deadline"},
]

VOLUME_MOUNT = "./local-data"
QUESTIONS_SUBDIR = "phase1_fsqa_eval"
SPLIT_DIR = "phase2_splits"
OUTPUT_SUBDIR = "phase3_generation"

ECFR_BASE = "https://www.ecfr.gov/api/versioner/v1/full"
ECFR_DATE = "2026-08-01"
ECFR_HEADERS = {"User-Agent": "labs-benchmark/1.0 (research; you@example.com)"}
CFR_TEXT_CAP = 15000

GEN_WORKERS = 6
JUDGE_WORKERS = 20
REQUEST_TIMEOUT = 600
MAX_RETRIES = 3
MAX_TOKENS = 32000
SEED = 20260825

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY missing.")
if not os.path.isdir(VOLUME_MOUNT):
    raise RuntimeError(f"{VOLUME_MOUNT} not mounted.")

HEADERS = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json",
           "X-OpenRouter-Title": "labs-benchmark"}
random.seed(SEED)

# --- Spend guard -----------------------------------------------------------
# Every OpenRouter call is reserved at its worst case before it is sent and settled after (guarded_post).
# A call is refused, and the run stops, if its model has no entry in PRICE, if its body has no max_tokens,
# or if its worst case would take the run past RUN_BUDGET_USD. PRICE holds each model's listed price in
# USD per million tokens (input, output). Entries the repo does not state are left for the operator to fill.
from datetime import datetime, timezone
import math


# --- Canonical env parsing and ledger-root rule (identical in every notebook) ---------------------------------
# RUN_BUDGET_USD (this notebook reads no other budget variable): unset means the default. Any value that is set, a blank or whitespace
# value included, must parse as a finite float greater than 0, or the notebook refuses to start.
def _env_usd(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return float(default)
    try:
        v = float(raw.strip())
    except ValueError:
        v = float("nan")
    if not math.isfinite(v) or v <= 0:
        raise RuntimeError(f"Refusing to start: {name} must be a finite number greater than 0 (got {raw!r}).")
    return v


RUN_BUDGET_USD = _env_usd("RUN_BUDGET_USD", 25.00)   # one cap per run; set the RUN_BUDGET_USD environment variable to change it
RUN_ID = None              # None = new run with a fresh ledger. Set a non-empty string to resume and keep its spend.
missing_prices = sorted(set([GENERATOR["model"]] + [JUDGES[j]["slug"] for j in USE_JUDGES]) - set(PRICE))
if missing_prices:
    raise RuntimeError(f"No confirmed price for {missing_prices}. Nothing was sent. Add the listed USD per million "
                       "tokens (input, output) for each model to PRICE in this config cell, then rerun.")
# None is the only signal for a new run. Any other value is a resume ID and goes through _checked_run_id below,
# which refuses an empty string, 0, False and every other non-string value. Nothing is defaulted.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
def _checked_run_id(value):
    """RUN_ID names one directory under SPEND_ROOT. Refuse an absolute path or any '..' segment, so the text of
    RUN_ID cannot lead os.path.join(SPEND_ROOT, RUN_ID) outside SPEND_ROOT. This is a string check only: it does
    not detect a symlink that already exists under SPEND_ROOT. Fails loudly."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Refusing to start: RUN_ID must be a non-empty string (got {value!r}).")
    if os.path.isabs(value) or value.startswith(("/", "\\")) or os.path.splitdrive(value)[0]:
        raise RuntimeError(f"Refusing to start: RUN_ID must be a relative name, not an absolute path (got {value!r}).")
    if ".." in value.replace("\\", "/").split("/"):
        raise RuntimeError(f"Refusing to start: RUN_ID must not contain a '..' segment (got {value!r}).")
    return value
RUN_ID = _checked_run_id(RUN_ID)
RUN_DIR = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)

# One ledger root and lock root for every notebook: SPEND_LEDGER_DIR, default <repo root>/spend_ledger. No .git above
# the working directory and no SPEND_LEDGER_DIR means refuse, with no fallback to the working directory.
# Ledger root. If SPEND_LEDGER_DIR is set at all, it is stripped: blank or whitespace-only refuses, anything else
# becomes os.path.abspath(os.path.expanduser(value)). If it is not set, the root is <repo root>/spend_ledger, where
# the repo root is the first directory, starting at the absolute working directory and walking up, for which
# os.path.exists(<dir>/.git) is true (a .git directory or a .git file). With no such directory the notebook refuses:
# there is no fallback to the working directory, because a different working directory would give a different lock.
# The lock is <root>/runner.lock and the run ledgers are <root>/<RUN_ID>/spend_ledger.jsonl.
def _ledger_root():
    raw = os.environ.get("SPEND_LEDGER_DIR")
    if raw is not None:
        value = raw.strip()
        if not value:
            raise RuntimeError("Refusing to start: SPEND_LEDGER_DIR is set but blank. Unset it to use "
                               "<repo root>/spend_ledger, or set it to a directory.")
        return os.path.abspath(os.path.expanduser(value))
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, ".git")):
            return os.path.join(d, "spend_ledger")
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Refusing to start: SPEND_LEDGER_DIR is not set and there is no .git above the working "
                               f"directory ({start}). Set SPEND_LEDGER_DIR to the shared ledger directory.")
        d = parent


SPEND_LEDGER_ROOT = _ledger_root()
LEDGER_DIR = os.path.join(SPEND_LEDGER_ROOT, RUN_ID)
os.makedirs(LEDGER_DIR, exist_ok=True)
LEDGER_PATH = os.path.join(LEDGER_DIR, "spend_ledger.jsonl")

# One paid runner at a time across all notebooks. A single exclusive lock file in the shared ledger root: a second
# runner refuses to start while it exists. It is removed on a clean exit only. A stale lock (a runner that died
# without exiting) is never cleared automatically: confirm no runner is alive, then delete the runner.lock file
# named in the error by hand to resume.
import atexit
RUNNER_LOCK = os.path.join(SPEND_LEDGER_ROOT, "runner.lock")
try:
    _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError:
    raise RuntimeError(f"{RUNNER_LOCK} exists. Another runner may be using this run, or a runner stopped without "
                       "releasing it. Refusing to start. Check for a live runner before deleting the lock by hand.")
_RUNNER_TOKEN = f"pid {os.getpid()} {datetime.now(timezone.utc).isoformat()}"
os.write(_lock_fd, _RUNNER_TOKEN.encode())
os.close(_lock_fd)


def _release_runner_lock():
    try:
        with open(RUNNER_LOCK) as f:
            if f.read() == _RUNNER_TOKEN:
                os.remove(RUNNER_LOCK)
    except OSError:
        pass


atexit.register(_release_runner_lock)


print(f"Generator : {GENERATOR['config_id']}")
print(f"Baseline  : Opus 5 pairs from set B, same seeds")
print(f"Judges    : {USE_JUDGES}")
print(f"Budget    : ${RUN_BUDGET_USD:.2f} per run   ledger: {LEDGER_PATH}")
print(f"Seeds     : {len(SEEDS)}")

## Load the Opus baseline

In [ ]:
def list_runs(subdir, required):
    base = os.path.join(VOLUME_MOUNT, subdir)
    return sorted(d for d in os.listdir(base)
                  if not d.startswith("_")
                  and os.path.isfile(os.path.join(base, d, required)))


rid = list_runs(QUESTIONS_SUBDIR, "questions.jsonl")[-1]
ALL_Q = [json.loads(l) for l in open(
    os.path.join(VOLUME_MOUNT, QUESTIONS_SUBDIR, rid, "questions.jsonl"))]
sp = json.load(open(os.path.join(VOLUME_MOUNT, SPLIT_DIR, f"split_{rid}.json")))
B_IDS = set(sp["set_b"])

seed_batches = {s["batch_id"] for s in SEEDS}
OPUS_PAIRS = [q for q in ALL_Q if q["id"] in B_IDS and q["batch_id"] in seed_batches]

print(f"Phase 1 run : {rid}")
print(f"Set B       : {len(B_IDS)}")
print(f"Opus baseline pairs: {len(OPUS_PAIRS)}")
for s in SEEDS:
    n = sum(1 for q in OPUS_PAIRS if q["batch_id"] == s["batch_id"])
    print(f"  {s['batch_id']:<12} 21 CFR {s['section']:<8} {s['archetype']:<32} n={n}")

EXPOSED_B_IDS = sorted(q["id"] for q in OPUS_PAIRS)
print(f"\nSet B ids exposed to judges here: {len(EXPOSED_B_IDS)}; exclude them from the final evaluation.")

## Fetch CFR text

In [ ]:
def fetch_cfr_section(section, retries=4):
    part = section.split(".")[0]
    url = f"{ECFR_BASE}/{ECFR_DATE}/title-21.xml"
    for attempt in range(retries):
        try:
            r = requests.get(url, params={"part": part, "section": section},
                             timeout=60, headers=ECFR_HEADERS)
            if r.status_code == 200:
                r.encoding = "utf-8"
                txt = re.sub(r"<[^>]+>", " ", r.text)
                txt = re.sub(r"\s+", " ", txt).strip()
                m = re.match(r"§\s*[\d.]+\s+(.+?\.)\s", txt)
                return (m.group(1).strip() if m else f"Section {section}"), txt
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** (attempt + 2))
                continue
            return None, None
        except Exception:
            time.sleep(2 ** attempt)
    return None, None


def questions_for_length(n):
    return 8 if n < 1000 else 12 if n < 2500 else 16 if n < 5000 else 20


CFR = {}
for s in SEEDS:
    print(f"  fetching {s['section']}", flush=True)
    h, t = fetch_cfr_section(s["section"])
    if t:
        CFR[s["section"]] = {"heading": h, "text": t, "chars": len(t),
                             "max_questions": questions_for_length(len(t))}
    time.sleep(0.3)

print(f"\nRetrieved {len(CFR)}/{len(SEEDS)}")
for s in SEEDS:
    v = CFR.get(s["section"])
    if v:
        print(f"  21 CFR {s['section']:<8} {v['chars']:>6}ch  ask={v['max_questions']:>2}  {v['heading'][:50]}")
target = sum(CFR[s["section"]]["max_questions"] for s in SEEDS if s["section"] in CFR)
print(f"\n5.5 will be asked for ~{target} pairs against {len(OPUS_PAIRS)} Opus pairs.")

## Checkpointing

In [ ]:
CKPT_DIR = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, "_checkpoint")
os.makedirs(CKPT_DIR, exist_ok=True)


class Checkpoint:
    def __init__(self, path, key_fn):
        self.path, self.key_fn = path, key_fn
        self.lock = threading.Lock()
        self.rows, self.done = [], set()
        if os.path.exists(path):
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        r = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    self.rows.append(r)
                    self.done.add(key_fn(r))
        self.fh = open(path, "a")

    def has(self, k):
        return k in self.done

    def add(self, row):
        with self.lock:
            self.fh.write(json.dumps(row) + "\n")
            self.fh.flush()
            os.fsync(self.fh.fileno())
            self.rows.append(row)
            self.done.add(self.key_fn(row))

    def close(self):
        try:
            self.fh.close()
        except Exception:
            pass


gen_ckpt = Checkpoint(os.path.join(CKPT_DIR, f"generated_{rid}.jsonl"),
                      lambda r: r["batch_id"])
judge_ckpt = Checkpoint(os.path.join(CKPT_DIR, f"grades_{rid}.jsonl"),
                        lambda r: (r["judge"], r["pair_id"]))
print(f"Generation checkpoint: {len(gen_ckpt.rows)} batches recorded")
print(f"Judge checkpoint     : {len(judge_ckpt.rows)} gradings recorded")

## Request helper

In [ ]:
import threading, uuid
from decimal import Decimal


class BudgetExceeded(RuntimeError):
    pass


PROMPT_OVERHEAD_TOKENS = 256    # allowance for chat-template tokens that the byte bound below does not cover


def max_price_for(model):
    """provider.max_price for one model: its listed price in USD per million tokens (input, output). PRICE already
    holds per-million figures, so the value is converted with Decimal and not scaled. An unpriced model is refused."""
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    pin, pout = PRICE[model]
    return {"prompt": float(Decimal(str(pin))), "completion": float(Decimal(str(pout)))}


def openrouter_body(body):
    """The body every OpenRouter call sends: the caller's body with provider.max_price set from PRICE and usage
    accounting switched on. Done here, once, so no call site can leave either out."""
    b = dict(body)
    b["provider"] = {**(body.get("provider") or {}), "max_price": max_price_for(body["model"])}
    b["usage"] = {"include": True}
    return b


def worst_case_cost(body):
    """Upper bound for one attempt. Input comes from the UTF-8 byte count of the messages (each token covers
    at least one byte); output is capped by max_tokens, which includes reasoning. Fails closed."""
    model = body["model"]
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    if not isinstance(body.get("max_tokens"), int):
        raise BudgetExceeded(f"not sent: request for {model} has no max_tokens, so its cost cannot be bounded.")
    pin, pout = PRICE[model]
    in_bound = len(json.dumps(body["messages"], ensure_ascii=False).encode("utf-8")) + PROMPT_OVERHEAD_TOKENS
    return pin / 1e6 * in_bound + pout / 1e6 * body["max_tokens"]


def _is_count(x):
    return isinstance(x, int) and not isinstance(x, bool) and x >= 0


def attempt_cost(body, usage):
    """Settle only against complete, validated provider usage: a reported cost that is finite and above zero,
    and both token counts as non-negative integers. Anything less keeps the worst case that was reserved, so a
    zero, missing, partial or malformed reply cannot release the reservation."""
    if isinstance(usage, dict):
        c = usage.get("cost")
        if (isinstance(c, (int, float)) and not isinstance(c, bool) and 0 < c < float("inf")
                and _is_count(usage.get("prompt_tokens")) and _is_count(usage.get("completion_tokens"))):
            return float(c), "usage"
    return worst_case_cost(body), "worst_case"


_ledger_lock = threading.Lock()
_ledger = {}                    # attempt id -> entry; reloaded from LEDGER_PATH on resume
_budget_halted = False          # set by a refused dispatch or by an overrun; no later attempt is sent
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH) as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                _ledger.setdefault(_rec["id"], {}).update(_rec)
    _budget_halted = any(_e.get("overrun") for _e in _ledger.values())   # an overrun halts the run after a restart too


def ledger_total():             # caller holds _ledger_lock. A reserved attempt with no settled line counts at worst case.
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())


def _ledger_write(rec):
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def ledger_reserve(body, purpose):
    """Reserve one attempt at its worst case before it is sent. Raises BudgetExceeded if the run is halted, or if
    the reservation would take the run past RUN_BUDGET_USD; landing exactly on the cap is allowed."""
    global _budget_halted
    wc = round(worst_case_cost(body), 6)
    with _ledger_lock:
        if _budget_halted:
            raise BudgetExceeded(f"not sent: the run is halted, so {body['model']} ({purpose}) is refused before dispatch")
        if ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "state": "reserved", "model": body["model"], "purpose": purpose,
                 "worst_case": wc, "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            _ledger_write(e)
            return aid
        _budget_halted = True
    raise BudgetExceeded(f"not sent: {body['model']} ({purpose}), worst case ${wc:.4f}, "
                         f"would take the run past RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")


def ledger_settle(aid, body, usage):
    """Settle one attempt. A reported charge above the reservation for this attempt is an overrun: it is written to
    the ledger with its reservation and halts the run, so every later dispatch is refused before any request is sent."""
    global _budget_halted
    cost, source = attempt_cost(body, usage)
    with _ledger_lock:
        e = _ledger[aid]
        charged = round(cost, 6)
        overrun = source == "usage" and charged > e["worst_case"]
        e.update(state="settled", cost=charged, cost_source=source)
        rec = {"id": aid, "state": "settled", "cost": charged, "cost_source": source}
        if overrun:
            e["overrun"] = True
            rec.update(overrun=True, reserved=e["worst_case"])
            _budget_halted = True
        _ledger_write(rec)
    if overrun:
        print(f"Ledger overrun: an attempt was charged ${charged:.4f} against a reservation of ${e['worst_case']:.4f}. "
              f"The run is halted; no later attempt will be sent.")
    return e["cost"]


def ledger_summary():
    with _ledger_lock:
        es, total = list(_ledger.values()), ledger_total()
    by = {}
    for e in es:
        n, c = by.get(e["purpose"], (0, 0.0))
        by[e["purpose"]] = (n + 1, c + (e["cost"] if e["state"] == "settled" else e["worst_case"]))
    lines = [f"Spend ledger: {len(es)} attempts, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}"]
    lines += [f"  {p:8s} attempts={n:5d}  charged=${c:.4f}" for p, (n, c) in sorted(by.items())]
    lines.append(f"  total charged ${total:.4f}   reserved, not settled={sum(e['state'] == 'reserved' for e in es)}")
    return "\n".join(lines)


def guarded_post(body, headers, timeout=REQUEST_TIMEOUT, purpose="call"):
    """One POST to OpenRouter. The attempt is reserved at its worst case before it is sent and settled after, so
    retries, exceptions and rejected calls are all counted. BudgetExceeded is raised before anything is sent."""
    body = openrouter_body(body)
    aid = ledger_reserve(body, purpose)
    usage = None
    try:
        r = requests.post(OPENROUTER_URL, headers=headers, json=body, timeout=timeout)
        if r.status_code == 200:
            try:
                usage = (r.json() or {}).get("usage") or None
            except Exception:
                usage = None
        return r
    finally:
        ledger_settle(aid, body, usage)


def call_openrouter(body, timeout=REQUEST_TIMEOUT, retries=MAX_RETRIES, retry_on_empty=False):
    last = None
    for attempt in range(retries):
        try:
            r = guarded_post(body, HEADERS, timeout=timeout, purpose="call")
            if r.status_code in (429, 503):
                w = r.headers.get("Retry-After")
                time.sleep(float(w) if w and w.isdigit() else 2 ** (attempt + 2))
                last = f"HTTP {r.status_code}"
                continue
            if r.status_code >= 400:
                last = f"HTTP {r.status_code}: {r.text[:300]}"
                if 400 <= r.status_code < 500 and r.status_code != 429:
                    return {"ok": False, "error": last}
                time.sleep(2 ** attempt)
                continue
            d = r.json()
            if "error" in d:
                return {"ok": False, "error": str(d["error"])[:300]}
            text = (d["choices"][0]["message"].get("content") or "").strip()
            if not text and retry_on_empty and attempt < retries - 1:
                time.sleep(1 + attempt)
                continue
            u = d.get("usage", {}) or {}
            return {"ok": True, "text": text, "provider": d.get("provider"),
                    "is_byok": u.get("is_byok"),
                    "prompt_tokens": u.get("prompt_tokens", 0),
                    "completion_tokens": u.get("completion_tokens", 0), "error": None}
        except BudgetExceeded:
            raise
        except Exception as e:
            last = f"{type(e).__name__}: {str(e)[:300]}"
            time.sleep(2 ** attempt)
    return {"ok": False, "error": last}


print("helper loaded")

## Generation — Phase 1's prompt, unchanged

Same system prompt, same user template, same schema Opus was given. The only
variable is the model.

In [ ]:
GEN_SYSTEM = """You are building an evaluation set for a food safety regulatory model. These questions will score several LLMs on FSQA reasoning; the winner then generates fine-tuning data. Bad questions produce a bad model, so precision matters more than volume.

The full text of the governing CFR section is provided to you. Write questions and answers from that text. Do not rely on memory for what the regulation says."""

GEN_TEMPLATE = """Generate up to {n} questions.

=== GOVERNING SECTION: 21 CFR {section} — {heading} ===
{section_text}
=== END SECTION ===

ARCHETYPE for this batch: {archetype}

Each question must:
- Be answerable from the section text above
- Be open-ended. No multiple choice, no yes/no
- Describe a real situation a QA manager or FSVP importer would face
- Use the words practitioners use, not regulatory boilerplate
- Be distinct from the others in this batch

Each reference answer must:
- Cite the specific subsection, e.g. "21 CFR {section}(b)(2)"
- Quote or closely track the operative language from the text above
- State the requirement plainly in under 120 words
- Note any condition or exception that changes the answer
- Contain nothing that is not in the section text above

Difficulty mix: roughly 30% straightforward recall, 50% requiring application to the scenario, 20% requiring reconciliation of two or more provisions within the section.

If the section does not support {n} distinct questions of this archetype, return fewer. Do not pad. Every citation must appear in the text above.

Return JSON only, matching the schema."""

GEN_SCHEMA = {
    "type": "json_schema",
    "json_schema": {"name": "fsqa_eval_batch", "strict": True,
        "schema": {"type": "object", "properties": {"questions": {"type": "array", "items": {
            "type": "object",
            "properties": {"question": {"type": "string"}, "reference_answer": {"type": "string"},
                           "citation": {"type": "string"},
                           "difficulty": {"type": "string",
                                          "enum": ["recall", "application", "reconciliation"]}},
            "required": ["question", "reference_answer", "citation", "difficulty"],
            "additionalProperties": False}}},
            "required": ["questions"], "additionalProperties": False}},
}

CIT = re.compile(r"21\s*CFR\s*\d+\.\d+(?:\([a-z0-9]+\))*")


def norm_citation(q):
    c = q.get("citation")
    if isinstance(c, str) and c.strip():
        return c
    cs = q.get("citations")
    if isinstance(cs, list) and cs:
        return "; ".join(str(x) for x in cs)
    if isinstance(cs, str):
        return cs
    m = CIT.search(q.get("reference_answer", ""))
    return m.group(0) if m else ""


def generate_batch(seed):
    sec = CFR[seed["section"]]
    body = {"model": GENERATOR["model"], "provider": EVAL_PROVIDER,
            "reasoning": {"effort": GENERATOR["effort"]},
            "max_tokens": MAX_TOKENS,
            "response_format": GEN_SCHEMA,
            "messages": [{"role": "system", "content": GEN_SYSTEM},
                         {"role": "user", "content": GEN_TEMPLATE.format(
                             n=sec["max_questions"], section=seed["section"],
                             heading=sec["heading"], section_text=sec["text"],
                             archetype=seed["archetype"])}]}
    res = call_openrouter(body)
    out = {"batch_id": seed["batch_id"], "section": seed["section"],
           "archetype": seed["archetype"], "domain": seed["domain"],
           "asked": sec["max_questions"], "ok": res["ok"], "error": res.get("error"),
           "prompt_tokens": res.get("prompt_tokens", 0),
           "completion_tokens": res.get("completion_tokens", 0), "questions": []}
    if res["ok"]:
        try:
            txt = res["text"].replace("```json", "").replace("```", "").strip()
            b = txt.find("{")
            if b > 0:
                txt = txt[b:]
            parsed = json.loads(txt)
            for i, q in enumerate(parsed.get("questions", [])):
                cite = norm_citation(q)
                if not CIT.search(cite) or seed["section"] not in cite:
                    continue
                out["questions"].append({
                    "id": f"gen-{seed['batch_id']}-{i+1:02d}",
                    "question": q.get("question", ""),
                    "reference_answer": q.get("reference_answer", ""),
                    "citation": cite,
                    "difficulty": q.get("difficulty", "application"),
                    "seed_section": seed["section"], "batch_id": seed["batch_id"],
                    "domain": seed["domain"], "seed_archetype": seed["archetype"]})
        except Exception as e:
            out["error"] = f"parse: {type(e).__name__}"
    gen_ckpt.add(out)
    print(f"  [{seed['batch_id']}] {len(out['questions'])}/{out['asked']} kept", flush=True)
    return out


todo = [s for s in SEEDS if not gen_ckpt.has(s["batch_id"]) and s["section"] in CFR]
print(f"Generating {len(todo)} batches with {GENERATOR['config_id']}...", flush=True)
_g0 = time.time()
if todo:
    with concurrent.futures.ThreadPoolExecutor(max_workers=GEN_WORKERS) as pool:
        list(pool.map(generate_batch, todo))
gen_elapsed = time.time() - _g0

GEN_PAIRS = [q for b in gen_ckpt.rows for q in b["questions"]]
_gp_in, _gp_out = PRICE[GENERATOR["model"]]
gen_cost = sum(b["prompt_tokens"] / 1e6 * _gp_in + b["completion_tokens"] / 1e6 * _gp_out
               for b in gen_ckpt.rows)
print(f"\nGenerated in {gen_elapsed/60:.1f} min")
print(f"  pairs : {len(GEN_PAIRS)}")
print(f"  cost  : ${gen_cost:.2f}")
print(f"  by seed: {dict(Counter(q['batch_id'] for q in GEN_PAIRS))}")

## Blind the pairs

Sources are stripped and the order shuffled. Judges see `pair_XXXX` and
nothing else. The mapping stays local and is written only after grading.

In [ ]:
POOL = []
for q in OPUS_PAIRS:
    POOL.append({**q, "_source": "opus"})
for q in GEN_PAIRS:
    POOL.append({**q, "_source": "gpt-5.5"})

rng = random.Random(SEED)
rng.shuffle(POOL)
for i, p in enumerate(POOL):
    p["pair_id"] = f"pair_{i:04d}"

BLIND_KEY = {p["pair_id"]: p["_source"] for p in POOL}

print(f"Pool: {len(POOL)} pairs")
print(f"  by source : {dict(Counter(p['_source'] for p in POOL))}")
print(f"  by section: {dict(Counter(p['seed_section'] for p in POOL))}")
print(f"\nFirst five as the judges will see them:")
for p in POOL[:5]:
    print(f"  {p['pair_id']}  21 CFR {p['seed_section']:<8} {p['question'][:60]}...")

## Grading

The rubric checks three separate things: is the question well-formed, is the
answer correct, and do they match. A pair can have an excellent answer to a
broken question.

In [ ]:
JUDGE_SYSTEM = """You are reviewing question-and-answer pairs written to train a food safety regulatory model. You are given the full text of the governing CFR section. That text is the ground truth.

Judge the PAIR, not just the answer.

question_quality (0-3): Is the question well-formed, answerable from this section alone, and grounded in a realistic situation a food safety professional would face?
  3 = realistic, specific, cleanly answerable
  2 = answerable but generic or artificial
  1 = vague, or requires information outside the section
  0 = malformed, or not answerable from this section

answer_accuracy (0-5): Is the reference answer correct as a statement of what this section requires?
  5 = correct and precise
  4 = correct, minor imprecision
  3 = substantially correct, one meaningful gap
  2 = a material error alongside correct content
  1 = mostly wrong
  0 = wrong

pair_coherence (0-2): Does the answer actually answer the question asked?
  2 = directly and completely
  1 = partially, or answers an adjacent question
  0 = does not answer it

citation_valid (0-2): Does the cited subsection exist in the text above and support the answer?
  2 = exists and supports it
  1 = exists but is the wrong subsection for this point
  0 = does not exist in the text

contradiction (boolean): Does the answer contradict the CFR text, or assert a requirement, deadline or threshold the section does not contain? Citing a related provision outside this section is not a contradiction.

Return JSON only:
{"question_quality": N, "answer_accuracy": N, "pair_coherence": N, "citation_valid": N, "contradiction": true|false, "note": "one short sentence"}"""

JUDGE_TEMPLATE = """=== GOVERNING CFR SECTION: 21 CFR {section} (ground truth) ===
{cfr_text}
=== END SECTION ===

PAIR {pair_id}

QUESTION:
{question}

REFERENCE ANSWER:
{answer}

STATED CITATION: {citation}

Grade the pair against the CFR text above. JSON only."""


def judge_pair(args):
    jn, p = args
    j = JUDGES[jn]
    body = {"model": j["slug"], "provider": j["provider"], "max_tokens": 4000,
            "messages": [{"role": "system", "content": JUDGE_SYSTEM},
                         {"role": "user", "content": JUDGE_TEMPLATE.format(
                             section=p["seed_section"],
                             cfr_text=CFR[p["seed_section"]]["text"][:CFR_TEXT_CAP],
                             pair_id=p["pair_id"], question=p["question"],
                             answer=p["reference_answer"], citation=p["citation"])}]}
    if j["reasoning"]:
        body["reasoning"] = j["reasoning"]
    res = call_openrouter(body, timeout=300, retry_on_empty=True)
    row = {"judge": jn, "pair_id": p["pair_id"], "seed_section": p["seed_section"],
           "domain": p["domain"], "batch_id": p["batch_id"]}
    if not res["ok"] or not res["text"]:
        row.update({"answer_accuracy": None, "error": res.get("error") or "empty"})
    else:
        try:
            txt = res["text"].replace("```json", "").replace("```", "").strip()
            b = txt.find("{")
            if b > 0:
                txt = txt[b:]
            g = json.loads(txt)
            qq, aa = int(g.get("question_quality", 0)), int(g.get("answer_accuracy", 0))
            pc, cv = int(g.get("pair_coherence", 0)), int(g.get("citation_valid", 0))
            row.update({"question_quality": qq, "answer_accuracy": aa,
                        "pair_coherence": pc, "citation_valid": cv,
                        "contradiction": bool(g.get("contradiction")),
                        "note": g.get("note", ""), "total": qq + aa + pc + cv,
                        "error": None})
        except Exception as e:
            row.update({"answer_accuracy": None,
                        "error": f"parse: {type(e).__name__} :: {res['text'][:120]}"})
    judge_ckpt.add(row)
    return row


jt = [(jn, p) for jn in USE_JUDGES for p in POOL
      if not judge_ckpt.has((jn, p["pair_id"]))]
print(f"Judge tasks: {len(USE_JUDGES) * len(POOL):,} total, {len(jt):,} remaining", flush=True)

_j0 = time.time(); _n = [0]


def _tj(t):
    r = judge_pair(t)
    _n[0] += 1
    if _n[0] % 50 == 0:
        el = time.time() - _j0
        print(f"  {_n[0]}/{len(jt)}  eta {(len(jt)-_n[0])/(_n[0]/el)/60:.0f}m", flush=True)
    return r


if jt:
    with concurrent.futures.ThreadPoolExecutor(max_workers=JUDGE_WORKERS) as pool:
        list(pool.map(_tj, jt))
judge_elapsed = time.time() - _j0

GRADES = judge_ckpt.rows
graded = [g for g in GRADES if g.get("answer_accuracy") is not None]
print(f"\nJudged in {judge_elapsed/60:.1f} min")
print(f"  parsed: {len(graded):,}/{len(GRADES):,}")
_b = [g for g in GRADES if g.get("answer_accuracy") is None]
if _b:
    print(f"  failed: {dict(Counter(g['judge'] for g in _b))}")

## Unblind and score

In [ ]:
for g in graded:
    g["source"] = BLIND_KEY.get(g["pair_id"], "?")

DIMS = ["question_quality", "answer_accuracy", "pair_coherence", "citation_valid", "total"]
MAXES = {"question_quality": 3, "answer_accuracy": 5, "pair_coherence": 2,
         "citation_valid": 2, "total": 12}

print("=" * 78)
print("HEAD TO HEAD — all judges pooled")
print("=" * 78)
print(f"{'source':<12}{'n':>6}" + "".join(f"{d.split('_')[0][:9]:>11}" for d in DIMS) + f"{'contra':>9}")
by_src = defaultdict(lambda: defaultdict(list))
for g in graded:
    for d in DIMS:
        by_src[g["source"]][d].append(g[d])
    by_src[g["source"]]["contra"].append(1 if g["contradiction"] else 0)
for src in ["opus", "gpt-5.5"]:
    v = by_src[src]
    if not v["total"]:
        continue
    line = f"{src:<12}{len(v['total']):>6}"
    line += "".join(f"{statistics.mean(v[d]):>11.3f}" for d in DIMS)
    line += f"{statistics.mean(v['contra'])*100:>8.1f}%"
    print(line)

print("\n" + "=" * 78)
print("PER JUDGE — does any judge favor its own family?")
print("=" * 78)
print(f"{'judge':<10}{'opus total':>13}{'5.5 total':>12}{'delta':>9}   (delta > 0 favors Opus)")
deltas = {}
for jn in USE_JUDGES:
    o = [g["total"] for g in graded if g["judge"] == jn and g["source"] == "opus"]
    f = [g["total"] for g in graded if g["judge"] == jn and g["source"] == "gpt-5.5"]
    if not o or not f:
        continue
    d = statistics.mean(o) - statistics.mean(f)
    deltas[jn] = d
    print(f"{jn:<10}{statistics.mean(o):>13.3f}{statistics.mean(f):>12.3f}{d:>+9.3f}")

if len(deltas) >= 3:
    print("\nSelf-preference check:")
    others = [v for k, v in deltas.items() if k != "opus"]
    if "opus" in deltas and others:
        print(f"  Opus judging Opus     : {deltas['opus']:+.3f}   "
              f"other judges avg: {statistics.mean(others):+.3f}   "
              f"bias: {deltas['opus'] - statistics.mean(others):+.3f}")
    others = [v for k, v in deltas.items() if k != "sol"]
    if "sol" in deltas and others:
        print(f"  Sol judging GPT-5.5   : {deltas['sol']:+.3f}   "
              f"other judges avg: {statistics.mean(others):+.3f}   "
              f"bias: {deltas['sol'] - statistics.mean(others):+.3f}")
    print("  (negative bias for Sol = Sol scores 5.5 higher than others do)")

print("\n" + "=" * 78)
print("BY SECTION")
print("=" * 78)
bs = defaultdict(lambda: defaultdict(list))
for g in graded:
    bs[g["seed_section"]][g["source"]].append(g["total"])
print(f"{'section':<12}{'opus':>10}{'gpt-5.5':>10}{'delta':>9}")
for sec in sorted(bs):
    o, f = bs[sec]["opus"], bs[sec]["gpt-5.5"]
    if o and f:
        print(f"21 CFR {sec:<6}{statistics.mean(o):>10.2f}{statistics.mean(f):>10.2f}"
              f"{statistics.mean(o)-statistics.mean(f):>+9.2f}")

## Diversity

A model asked for 20 questions tends to write the same twelve twelve ways.
Scoring cannot catch that. Jaccard overlap on content words, computed here,
no API calls.

In [ ]:
STOP = set("the a an of to in for and or is are be if what when must may you your "
           "this that with from on at as it its under section cfr does do can".split())


def toks(s):
    return {w for w in re.findall(r"[a-z]{3,}", s.lower()) if w not in STOP}


def mean_pairwise_jaccard(items):
    ts = [toks(x) for x in items]
    sims = []
    for a, b in itertools.combinations(range(len(ts)), 2):
        u = ts[a] | ts[b]
        if u:
            sims.append(len(ts[a] & ts[b]) / len(u))
    return statistics.mean(sims) if sims else 0.0


print(f"{'source':<12}{'batch':<12}{'n':>4}{'mean similarity':>18}")
div = defaultdict(list)
for src, pairs in [("opus", OPUS_PAIRS), ("gpt-5.5", GEN_PAIRS)]:
    bb = defaultdict(list)
    for p in pairs:
        bb[p["batch_id"]].append(p["question"])
    for b in sorted(bb):
        if len(bb[b]) < 2:
            continue
        s = mean_pairwise_jaccard(bb[b])
        div[src].append(s)
        print(f"{src:<12}{b:<12}{len(bb[b]):>4}{s:>18.3f}")
print()
for src in ["opus", "gpt-5.5"]:
    if div[src]:
        print(f"{src:<12} mean within-batch similarity: {statistics.mean(div[src]):.3f}  "
              f"(lower = more diverse)")

## Write outputs

In [ ]:
run_id = RUN_ID
outdir = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, run_id)
os.makedirs(outdir, exist_ok=True)


def dump(name, data):
    p = os.path.join(outdir, name)
    with open(p, "w") as f:
        for r in data:
            f.write(json.dumps(r) + "\n")
    return p


summary = {}
for src in ["opus", "gpt-5.5"]:
    v = by_src[src]
    if v["total"]:
        summary[src] = {d: round(statistics.mean(v[d]), 3) for d in DIMS}
        summary[src]["contradiction_rate"] = round(statistics.mean(v["contra"]), 4)
        summary[src]["n_gradings"] = len(v["total"])
        summary[src]["within_batch_similarity"] = (
            round(statistics.mean(div[src]), 4) if div[src] else None)

manifest = {
    "run_id": run_id, "stage": "3_generation",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "question_run_id": rid,
    "generator": GENERATOR, "judges": USE_JUDGES,
    "seeds": SEEDS,
    "opus_baseline_pairs": len(OPUS_PAIRS),
    "generated_pairs": len(GEN_PAIRS),
    "pool_size": len(POOL),
    "blind": True,
    "generation_cost_usd": round(gen_cost, 2),
    "judge_calls": len(GRADES), "judge_ok": len(graded),
    "judge_failures": dict(Counter(g["judge"] for g in GRADES
                                   if g.get("answer_accuracy") is None)),
    "summary": summary,
    "judge_deltas": {k: round(v, 4) for k, v in deltas.items()},
    "exposed_set_b_ids": EXPOSED_B_IDS,
    "elapsed_min": {"generate": round(gen_elapsed / 60, 1),
                    "judge": round(judge_elapsed / 60, 1)},
    "files": {"generated": dump("generated_pairs.jsonl", GEN_PAIRS),
              "pool": dump("pool.jsonl", POOL),
              "grades": dump("grades.jsonl", graded)},
}
with open(os.path.join(outdir, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)
with open(os.path.join(outdir, "blind_key.json"), "w") as f:
    json.dump(BLIND_KEY, f, indent=2)

gen_ckpt.close()
judge_ckpt.close()

print(f"Generation cost: ${manifest['generation_cost_usd']:.2f}")
print(f"Output: {outdir}")
print(f"\n{len(EXPOSED_B_IDS)} set B ids were shown to judges. Exclude them from the final evaluation:")
print(f"  see manifest['exposed_set_b_ids']")

## Inspect

Read the worst pairs from each source. A judge misreading a well-formed
question shows up here and nowhere else.

In [ ]:
def show(source=None, n=3, worst=True):
    pk = {p["pair_id"]: p for p in POOL}
    pool_ = [g for g in graded if source is None or g["source"] == source]
    pool_ = sorted(pool_, key=lambda g: g["total"], reverse=not worst)
    seen = set()
    shown = 0
    for g in pool_:
        if g["pair_id"] in seen or shown >= n:
            continue
        seen.add(g["pair_id"])
        p = pk[g["pair_id"]]
        print("=" * 78)
        print(f"{g['pair_id']} | source={g['source']} | judge={g['judge']} | total={g['total']}/12")
        print(f"  qq={g['question_quality']}/3 acc={g['answer_accuracy']}/5 "
              f"coh={g['pair_coherence']}/2 cite={g['citation_valid']}/2 "
              f"contra={g['contradiction']}")
        print(f"  note: {g.get('note')}")
        print(f"\nQ: {p['question'][:300]}")
        print(f"A: {p['reference_answer'][:350]}")
        print(f"CITE: {p['citation']}\n")
        shown += 1


if graded:
    print("WORST OPUS PAIRS\n")
    show(source="opus", n=2)
    print("\n\nWORST GPT-5.5 PAIRS\n")
    show(source="gpt-5.5", n=2)